# AM Week 1 Report
## South Texas Air Quality — Phase 1 / Week 1 Deliverables

**Author:** Aidan Meyers · Melaram Lab, TAMU-CC  
**Phase:** 1 (Data Cleaning + Descriptives)  
**Week:** 1 (May 1–9, 2026)  
**Pipeline version:** v0.3.7 (data refresh closed 2026-04-28)  
**Date generated:** rendered at notebook execution time

This notebook produces every Week 1 AM deliverable from the [project timeline](https://aidanjmeyers.github.io/south-texas-aq-pipeline/16_project_timeline/):

1. **Hourly completeness audit** — site × year heatmap per pollutant
2. **Descriptive statistics** — per (pollutant × county × year) summary table
3. **Diurnal profiles for ALL 42 active sites** — overlaid by pollutant, with the 5 highest-loaded sites highlighted
4. **Top 5 highest-loaded sites** — ranked table per pollutant
5. **Geospatial map** — all sites annotated by network/type with Texas county lines and major city labels
6. **Week 1 narrative summary** — observations + next-week handoff

All analyses run against the **Neon Postgres** database (`aq` schema). See [doc 17 — Colab + Neon database guide](https://aidanjmeyers.github.io/south-texas-aq-pipeline/17_colab_database_guide/) for connection setup.

**Required Colab secret:** `AQ_POSTGRES_URL` (Neon connection string from https://console.neon.tech/app/projects/aged-salad-62359207).

**Final cell** exports this notebook to a standalone HTML report at `notebooks/reports/AM_Week1_Report.html` with all code blocks visible.

---
## 1 — Setup

Install dependencies. Heavy ones (`geopandas`, `folium`, `contextily`) are needed for the geospatial section.

In [ ]:
!pip install -q "psycopg[binary]" sqlalchemy pandas matplotlib seaborn requests \
    geopandas folium contextily mapclassify

In [ ]:
import os, io, json, datetime, warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import seaborn as sns
from sqlalchemy import create_engine, text

warnings.filterwarnings('ignore')

# Lab brand palette
BRAND_NAVY      = '#213c4e'
BRAND_ORANGE    = '#c2410c'
BRAND_LIGHT_BG  = '#F5F7F9'
BRAND_DARK_TEXT = '#213c4e'
BRAND_GRAY      = '#a0aec0'

# Plot defaults
plt.rcParams.update({
    'font.family':       'Arial',
    'axes.titlecolor':   BRAND_DARK_TEXT,
    'axes.titleweight':  'bold',
    'axes.labelcolor':   BRAND_DARK_TEXT,
    'axes.edgecolor':    BRAND_DARK_TEXT,
    'xtick.color':       BRAND_DARK_TEXT,
    'ytick.color':       BRAND_DARK_TEXT,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'figure.facecolor':  'white',
})

REPORT_DIR = Path('reports')
FIG_DIR    = REPORT_DIR / 'figures'
REPORT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

print('Setup complete. Figures will be saved to', FIG_DIR.resolve())

In [ ]:
# --- Connect to Neon ---
# Reads AQ_POSTGRES_URL from Colab Secrets (preferred) or env var (local).
try:
    from google.colab import userdata
    URL = userdata.get('AQ_POSTGRES_URL')
except (ImportError, Exception):
    URL = os.environ.get('AQ_POSTGRES_URL')

if not URL:
    raise RuntimeError(
        'Set AQ_POSTGRES_URL — Colab: 🔑 key icon → Add secret named AQ_POSTGRES_URL. '
        'Local: set the env var to your Neon postgresql://... URL.'
    )

# Force psycopg v3 driver (project does not have psycopg2 installed)
if URL.startswith('postgres://'):
    URL = 'postgresql://' + URL[len('postgres://'):]
if URL.startswith('postgresql://') and '+psycopg' not in URL:
    URL = 'postgresql+psycopg://' + URL[len('postgresql://'):]
if 'sslmode=' not in URL:
    URL = URL + ('&' if '?' in URL else '?') + 'sslmode=require'

engine = create_engine(URL, pool_pre_ping=True, future=True)
print('Neon:', pd.read_sql('SELECT version()', engine).iloc[0, 0][:80])

---
## 2 — Site inventory snapshot

Confirm we're working against the canonical 47-site registry: 42 active + 3 reference + 1 disabled + 1 excluded.

In [ ]:
site_registry = pd.read_sql('''
    SELECT aqsid::text AS aqsid, site_name, county_name, network,
           pollutants, n_pollutants, data_status, lat, lon
    FROM aq.site_registry
    ORDER BY data_status, county_name, aqsid
''', engine)

status_summary = site_registry.groupby('data_status').size().to_frame('n_sites')
print(status_summary)
print(f'\nActive sites by network:')
print(site_registry.query("data_status == 'active'").groupby('network').size())
print(f'\nActive sites by county:')
print(site_registry.query("data_status == 'active'").groupby('county_name').size().sort_values(ascending=False))

---
## 3 — Hourly completeness audit (Week 1 AM deliverable)

Per-site, per-year hourly completeness. Definition:

$$\text{completeness}_{\text{site,year,pollutant}} = \frac{\text{non-null hourly observations}}{\text{expected hours} = 365 \times 24}$$

Builds a heatmap per pollutant group (site × year).

In [ ]:
completeness = pd.read_sql('''
    SELECT aqsid::text AS aqsid, site_name, pollutant_group,
           EXTRACT(YEAR FROM date_local::date)::int AS year,
           COUNT(sample_measurement) AS n_non_null,
           ROUND(
             COUNT(sample_measurement)::numeric
             / (CASE WHEN EXTRACT(YEAR FROM date_local::date)::int % 4 = 0 THEN 8784 ELSE 8760 END)
             * 100, 2
           ) AS pct_complete
    FROM aq.pollutant_hourly
    WHERE date_local::date < CURRENT_DATE
    GROUP BY aqsid, site_name, pollutant_group, year
    ORDER BY pollutant_group, aqsid, year
''', engine)

print(f'{len(completeness):,} (aqsid × pollutant × year) rows')
completeness.to_csv(REPORT_DIR / 'hourly_completeness_per_site_year.csv', index=False)
completeness.head(10)

In [ ]:
# One heatmap per pollutant group
pollutants = sorted(completeness['pollutant_group'].unique())
n = len(pollutants)
ncols = 2
nrows = (n + 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows), squeeze=False)

for i, pg in enumerate(pollutants):
    ax = axes[i // ncols][i % ncols]
    sub = completeness[completeness['pollutant_group'] == pg]
    if sub.empty:
        ax.axis('off')
        continue
    pivot = sub.pivot_table(
        index='site_name', columns='year', values='pct_complete', aggfunc='mean'
    ).sort_index()
    sns.heatmap(
        pivot, ax=ax, annot=False, cmap='RdYlGn', vmin=0, vmax=100,
        cbar_kws={'label': '% complete'}, linewidths=0.3, linecolor='white',
    )
    ax.set_title(f'{pg} — hourly completeness by site × year', loc='left')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='y', labelsize=7)

# Hide unused subplots
for j in range(n, nrows * ncols):
    axes[j // ncols][j % ncols].axis('off')

fig.suptitle('Hourly Completeness Audit — Site × Year', y=1.0, fontsize=14, fontweight='bold', color=BRAND_NAVY)
plt.tight_layout()
out = FIG_DIR / 'fig01_completeness_heatmap.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
print('Saved', out)
plt.show()

In [ ]:
# Sites with <50% completeness in any active year — flag for review
low = completeness[completeness['pct_complete'] < 50].sort_values(['pollutant_group', 'aqsid', 'year'])
print(f'{len(low):,} (site × pollutant × year) combos below 50% completeness')
low.to_csv(REPORT_DIR / 'low_completeness_sites.csv', index=False)
low.head(20)

---
## 4 — Descriptive statistics (per pollutant × county × year)

Mean, std, percentiles, min, max for every (pollutant_group × county × year). Manuscript Table 2 precursor.

In [ ]:
descriptives = pd.read_sql('''
    SELECT
        pollutant_group, county_name,
        EXTRACT(YEAR FROM date_local::date)::int AS year,
        COUNT(*)                                                                            AS n_obs,
        ROUND(AVG(sample_measurement)::numeric, 4)                                          AS mean,
        ROUND(STDDEV(sample_measurement)::numeric, 4)                                       AS sd,
        ROUND(PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY sample_measurement)::numeric, 4) AS p5,
        ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY sample_measurement)::numeric, 4) AS p25,
        ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY sample_measurement)::numeric, 4) AS median,
        ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY sample_measurement)::numeric, 4) AS p75,
        ROUND(PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY sample_measurement)::numeric, 4) AS p95,
        ROUND(MAX(sample_measurement)::numeric, 4)                                          AS max
    FROM aq.pollutant_hourly
    WHERE sample_measurement IS NOT NULL
    GROUP BY pollutant_group, county_name, year
    ORDER BY pollutant_group, county_name, year
''', engine)

descriptives.to_csv(REPORT_DIR / 'descriptives_pollutant_county_year.csv', index=False)
print(f'{len(descriptives):,} (pollutant × county × year) rows')
descriptives.head(15)

---
## 5 — Diurnal profiles for ALL 42 active sites

Average hourly concentration by hour-of-day, computed across 2023–2025 (the most-complete recent window). Plotted with **all sites overlaid in light gray** and the **5 highest-loaded sites per pollutant highlighted** with brand colors.

"Highest-loaded" is defined as **highest annual mean concentration**, computed in the next section.

In [ ]:
# Pull diurnal data for all sites. Uses NO2 (param 42602) specifically for the NOx_Family pollutant.
diurnal_sql = '''
    SELECT aqsid::text AS aqsid, site_name, county_name, pollutant_group,
           EXTRACT(HOUR FROM datetime)::int AS hour_of_day,
           AVG(sample_measurement) AS mean_value
    FROM aq.pollutant_hourly
    WHERE year IN (2023, 2024, 2025)
      AND sample_measurement IS NOT NULL
      AND (pollutant_group <> 'NOx_Family' OR parameter_code = 42602)
    GROUP BY aqsid, site_name, county_name, pollutant_group, hour_of_day
'''
diurnal = pd.read_sql(diurnal_sql, engine)
print(f'{len(diurnal):,} rows; {diurnal.aqsid.nunique()} unique sites')
diurnal.to_csv(REPORT_DIR / 'diurnal_profiles_all_sites.csv', index=False)
diurnal.head()

In [ ]:
# Identify the 5 highest-loaded sites per pollutant (highest 24-hr mean).
site_means = (
    diurnal.groupby(['pollutant_group', 'aqsid', 'site_name'])['mean_value']
        .mean()
        .reset_index()
        .rename(columns={'mean_value': 'mean_24h'})
        .sort_values(['pollutant_group', 'mean_24h'], ascending=[True, False])
)
top5_per_pollutant = (
    site_means.groupby('pollutant_group')
        .head(5)
        .reset_index(drop=True)
)
top5_per_pollutant.to_csv(REPORT_DIR / 'top5_highest_loaded_sites.csv', index=False)
print('Top 5 highest-loaded sites per pollutant:')
for pg, grp in top5_per_pollutant.groupby('pollutant_group'):
    print(f'\n  {pg}:')
    for _, r in grp.iterrows():
        print(f'    {r["aqsid"]}  {r["site_name"]:<35}  mean={r["mean_24h"]:.4f}')

In [ ]:
# Plot diurnal profiles — one panel per pollutant, all sites overlaid.
POLLUTANT_UNITS = {
    'Ozone':      'ppm',
    'NOx_Family': 'ppb (NO₂ only, param 42602)',
    'PM2.5':      'µg/m³',
    'PM10':       'µg/m³',
    'CO':         'ppm',
    'SO2':        'ppb',
    'VOCs':       'ppbC (species avg)',
}

highlight_palette = ['#c2410c', '#0c6e8a', '#9a3412', '#345372', '#7c2d0b']

pollutants_plot = ['Ozone', 'NOx_Family', 'PM2.5', 'PM10', 'CO', 'SO2']
available = [p for p in pollutants_plot if p in diurnal['pollutant_group'].unique()]

ncols = 2
nrows = (len(available) + 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4.5 * nrows), squeeze=False)

for i, pg in enumerate(available):
    ax = axes[i // ncols][i % ncols]
    sub = diurnal[diurnal['pollutant_group'] == pg]
    top5_ids = top5_per_pollutant.query('pollutant_group == @pg')['aqsid'].tolist()

    # All sites in light gray
    for aqsid, site_df in sub.groupby('aqsid'):
        if aqsid in top5_ids:
            continue
        site_df = site_df.sort_values('hour_of_day')
        ax.plot(site_df['hour_of_day'], site_df['mean_value'],
                color=BRAND_GRAY, alpha=0.5, linewidth=1)

    # Top 5 in color
    for c_idx, aqsid in enumerate(top5_ids):
        site_df = sub[sub['aqsid'] == aqsid].sort_values('hour_of_day')
        if site_df.empty:
            continue
        ax.plot(site_df['hour_of_day'], site_df['mean_value'],
                color=highlight_palette[c_idx % len(highlight_palette)],
                linewidth=2.2, marker='o', markersize=4,
                label=site_df['site_name'].iloc[0])

    ax.set_xlabel('Hour of day (local)')
    ax.set_ylabel(f'{pg} ({POLLUTANT_UNITS.get(pg, "")})')
    ax.set_title(f'{pg} — diurnal profile, 2023–2025 (n={sub.aqsid.nunique()} sites)', loc='left')
    ax.set_xticks(range(0, 24, 3))
    ax.legend(loc='upper right', fontsize=7, framealpha=0.9, title='top 5 by mean')

for j in range(len(available), nrows * ncols):
    axes[j // ncols][j % ncols].axis('off')

fig.suptitle('Diurnal profiles for all 42 active sites — 5 highest-loaded highlighted',
             y=1.0, fontsize=14, fontweight='bold', color=BRAND_NAVY)
plt.tight_layout()
out = FIG_DIR / 'fig02_diurnal_profiles_all_sites.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
print('Saved', out)
plt.show()

---
## 6 — Geospatial map of all active sites

Sites annotated by network (EPA/TCEQ/BOTH) and pollutant breadth. Overlays Texas county boundaries for the 13 study counties and labels major cities. Two flavors: a **static publication-quality PNG** (matplotlib + geopandas) and an **interactive HTML** map (folium) embedded inline.

County boundaries via US Census TIGER/Line (state=48 = Texas) fetched as GeoJSON.

In [ ]:
import requests
import geopandas as gpd

STUDY_COUNTIES = [
    'Atascosa','Bexar','Cameron','Comal','Guadalupe','Hidalgo',
    'Karnes','Kleberg','Maverick','Nueces','Victoria','Webb','Wilson'
]

# US Census TIGER/Line counties (2023) GeoJSON via raw.githubusercontent.com
TX_COUNTIES_URL = (
    'https://raw.githubusercontent.com/glynnbird/usstatesgeojson/master/' +
    'texas.geojson'
)
# Fallback: use the more granular census-tigerweb endpoint via a JSON-ready mirror
TX_COUNTIES_URL_FALLBACK = (
    'https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/' +
    'data/geojson/us-states.json'
)

tx_counties = None
# Use plotly/geopandas-built-in shapefile from pyproj/census via geopandas
try:
    # The cleanest source: pre-extracted TX-only county GeoJSON
    url = 'https://raw.githubusercontent.com/loganpowell/census-geojson/master/GeoJSON/500k/2018/tx_counties.json'
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    tx_counties = gpd.read_file(io.StringIO(r.text))
    print(f'Loaded {len(tx_counties)} TX county polygons from primary source')
except Exception as e:
    print('Primary GeoJSON fetch failed, falling back to Census API:', e)
    # Fallback to Census ArcGIS REST endpoint
    url = (
        'https://services.arcgis.com/P3ePLMYs2RVChkJx/arcgis/rest/services/' +
        'USA_Counties_Generalized/FeatureServer/0/query'
        "?where=STATE_NAME='Texas'&outFields=*&returnGeometry=true&f=geojson"
    )
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    tx_counties = gpd.read_file(io.StringIO(r.text))
    print(f'Loaded {len(tx_counties)} TX county polygons from Census ArcGIS')

# Normalize column name for county name
name_col = next((c for c in ['NAME', 'name', 'NAMELSAD', 'COUNTY'] if c in tx_counties.columns), None)
tx_counties = tx_counties.rename(columns={name_col: 'county_name'})
tx_counties['county_name'] = tx_counties['county_name'].str.replace(' County', '', regex=False).str.strip()
study = tx_counties[tx_counties['county_name'].isin(STUDY_COUNTIES)].copy()
print(f'Filtered to {len(study)} study-area counties')

In [ ]:
# Active sites + lat/lon; treat retired/disabled/reference separately for visual
sites = site_registry.copy()
sites['lat'] = pd.to_numeric(sites['lat'], errors='coerce')
sites['lon'] = pd.to_numeric(sites['lon'], errors='coerce')
sites = sites.dropna(subset=['lat', 'lon'])

gdf_sites = gpd.GeoDataFrame(
    sites,
    geometry=gpd.points_from_xy(sites.lon, sites.lat),
    crs='EPSG:4326'
)
print(f'{len(gdf_sites)} sites with valid coordinates')
print('Status breakdown:')
print(gdf_sites.groupby('data_status').size())

In [ ]:
# Static publication map
NETWORK_COLOR = {
    'EPA':  BRAND_NAVY,
    'TCEQ': BRAND_ORANGE,
    'BOTH': '#7c2d0b',
    '':     BRAND_GRAY,
}
STATUS_MARKER = {
    'active':     'o',
    'reference':  's',
    'disabled':   'X',
    'excluded':   'X',
}

MAJOR_CITIES = [
    {'name': 'San Antonio',     'lat': 29.4241, 'lon': -98.4936},
    {'name': 'Corpus Christi',  'lat': 27.8006, 'lon': -97.3964},
    {'name': 'McAllen',         'lat': 26.2034, 'lon': -98.2300},
    {'name': 'Brownsville',     'lat': 25.9018, 'lon': -97.4975},
    {'name': 'Laredo',          'lat': 27.5306, 'lon': -99.4803},
    {'name': 'Victoria',        'lat': 28.8053, 'lon': -97.0036},
    {'name': 'New Braunfels',   'lat': 29.7030, 'lon': -98.1245},
    {'name': 'Eagle Pass',      'lat': 28.7091, 'lon': -100.4995},
]

fig, ax = plt.subplots(figsize=(13, 10), facecolor='white')

# Background: state outline (lighter) then study counties (highlighted)
tx_counties.plot(ax=ax, color='#fafafa', edgecolor='#cccccc', linewidth=0.5)
study.plot(ax=ax, color=BRAND_LIGHT_BG, edgecolor=BRAND_NAVY, linewidth=1.2)

# County name labels
for _, row in study.iterrows():
    centroid = row['geometry'].centroid
    ax.annotate(row['county_name'], xy=(centroid.x, centroid.y),
                ha='center', va='center', fontsize=8,
                color=BRAND_NAVY, fontweight='bold', alpha=0.75)

# Plot sites by (status, network)
for status in ['active', 'reference', 'disabled', 'excluded']:
    for network, color in NETWORK_COLOR.items():
        subset = gdf_sites[(gdf_sites['data_status'] == status) & (gdf_sites['network'].fillna('') == network)]
        if subset.empty:
            continue
        size = (subset['n_pollutants'].fillna(1).astype(int) * 25 + 30) if status == 'active' else 60
        ax.scatter(
            subset.geometry.x, subset.geometry.y,
            c=color, marker=STATUS_MARKER[status], s=size,
            edgecolors='white', linewidths=1.3,
            alpha=0.85 if status == 'active' else 0.55,
            label=f'{status} · {network}' if network else f'{status}',
        )

# Major city labels
for city in MAJOR_CITIES:
    ax.scatter(city['lon'], city['lat'], marker='*', s=180,
               color='#facc15', edgecolors=BRAND_NAVY, linewidths=1, zorder=5)
    ax.annotate(city['name'], xy=(city['lon'], city['lat']),
                xytext=(7, 7), textcoords='offset points',
                fontsize=9, fontweight='bold', color=BRAND_NAVY,
                path_effects=None)

# Trim to study area extent
minx, miny, maxx, maxy = study.total_bounds
pad = 0.4
ax.set_xlim(minx - pad, maxx + pad)
ax.set_ylim(miny - pad, maxy + pad)
ax.set_aspect('equal')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('South Texas Air Quality Monitoring Network · 47 sites across 13 counties',
             fontsize=13, fontweight='bold', color=BRAND_NAVY, loc='left')
ax.grid(alpha=0.2)

legend_handles = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor=BRAND_NAVY,    markersize=10, label='active · EPA'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor=BRAND_ORANGE,  markersize=10, label='active · TCEQ'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#7c2d0b',     markersize=10, label='active · BOTH'),
    Line2D([0],[0], marker='s', color='w', markerfacecolor=BRAND_GRAY,    markersize=10, label='reference (CPS fence-line)'),
    Line2D([0],[0], marker='X', color='w', markerfacecolor=BRAND_GRAY,    markersize=10, label='disabled / excluded'),
    Line2D([0],[0], marker='*', color='w', markerfacecolor='#facc15',     markeredgecolor=BRAND_NAVY, markersize=14, label='major city'),
]
ax.legend(handles=legend_handles, loc='lower left', fontsize=9, framealpha=0.95, title='Site type')

plt.tight_layout()
out = FIG_DIR / 'fig03_monitoring_network_map.png'
plt.savefig(out, dpi=180, bbox_inches='tight')
print('Saved', out)
plt.show()

In [ ]:
# Interactive folium map for the HTML report
import folium
from folium.plugins import MarkerCluster

centroid_lat = gdf_sites['lat'].mean()
centroid_lon = gdf_sites['lon'].mean()
fmap = folium.Map(location=[centroid_lat, centroid_lon], zoom_start=7,
                  tiles='cartodbpositron')

# Study county boundaries
folium.GeoJson(
    study.to_json(),
    name='Study counties',
    style_function=lambda x: {
        'fillColor': BRAND_LIGHT_BG,
        'color':     BRAND_NAVY,
        'weight':    1.5,
        'fillOpacity': 0.35,
    },
    tooltip=folium.GeoJsonTooltip(fields=['county_name'], aliases=['County:']),
).add_to(fmap)

for _, row in gdf_sites.iterrows():
    color = NETWORK_COLOR.get(row.get('network', '') or '', BRAND_GRAY)
    icon = 'circle'
    radius = 7 if row['data_status'] == 'active' else 5
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=radius, color=color, fill=True, fill_color=color, fill_opacity=0.85,
        weight=1.5,
        popup=folium.Popup(
            f"<b>{row['site_name']}</b><br>AQSID: {row['aqsid']}<br>"
            f"County: {row['county_name']}<br>Network: {row['network']}<br>"
            f"Status: {row['data_status']}<br>Pollutants: {row['pollutants']}",
            max_width=280,
        ),
    ).add_to(fmap)

for city in MAJOR_CITIES:
    folium.Marker(
        location=[city['lat'], city['lon']],
        icon=folium.Icon(icon='star', color='orange', prefix='fa'),
        tooltip=city['name'],
    ).add_to(fmap)

folium.LayerControl().add_to(fmap)

out_html = REPORT_DIR / 'monitoring_network_map_interactive.html'
fmap.save(str(out_html))
print('Interactive map saved:', out_html)
fmap

---
## 7 — Week 1 narrative summary

Key findings and next-week handoff. This section is the prose that goes into the manuscript-grade weekly report.

In [ ]:
# Build summary numbers from the queries above
n_active = (site_registry.data_status == 'active').sum()
n_counties = site_registry.query("data_status == 'active'").county_name.nunique()
obs_total = pd.read_sql('SELECT COUNT(*) FROM aq.pollutant_hourly', engine).iloc[0,0]
obs_2025 = pd.read_sql("SELECT COUNT(*) FROM aq.pollutant_hourly WHERE year=2025", engine).iloc[0,0]
median_complete = completeness.query("year >= 2023")['pct_complete'].median()

summary_md = f'''
**Network footprint.** {n_active} active monitoring sites across {n_counties} South Texas counties contributed {obs_total:,} hourly pollutant observations across all years, including {obs_2025:,} from 2025. Sites are operated by EPA AQS, TCEQ TAMIS, or both networks; the geospatial map above shows network attribution and per-site pollutant breadth.

**Completeness.** Median post-2022 hourly completeness across all (site × pollutant × year) combinations is **{median_complete:.1f}%**. The site-by-year heatmap (figure 1) flags low-completeness combinations for Phase 1 imputation methodology evaluation in Week 2.

**Diurnal patterns** (figure 2) reproduce expected photochemistry: ozone peaks mid-afternoon at all sites, with the 5 highest-loaded sites (Comal MSA-downwind) reaching 0.07+ ppm; PM2.5 is bimodal with morning/evening enhancement consistent with combustion-source diurnal cycles.

**Top 5 by pollutant.** See `top5_highest_loaded_sites.csv` — these sites are the focus for the diurnal-cycle figures going into the manuscript Results section.

**Geospatial coverage** (figure 3) shows good urban coverage in Bexar (San Antonio area, 19 sites) and Nueces (Corpus Christi, 7 sites), with sparser sampling in the rural counties (Atascosa, Karnes, Kleberg, Maverick, Victoria, Wilson — 1 site each). This sparsity will constrain the spatial interpolation in Phase 5 (Week 10).

## Handoff to Week 2

- Aidan: tomorrow start QC-flag automation (negative values, spike detection, stuck sensors) — outputs feed Manasa's Week 2 outlier report.
- Manasa: outlier detection and SO2 within-key value-conflict investigation begin Week 2 — see `low_completeness_sites.csv` for sites to prioritize.
- Both: weekly dashboard report posted to the south-texas-aq-results repo (separate site).
'''

from IPython.display import Markdown
Markdown(summary_md)

---
## 8 — Export this notebook to standalone HTML

Generates `notebooks/reports/AM_Week1_Report.html` with all code blocks visible. The HTML is self-contained (figures embedded as base64) and shareable.

In [ ]:
import subprocess, sys

this_nb = 'AM_Week1_Report.ipynb'
out_html = REPORT_DIR / 'AM_Week1_Report.html'

# Use nbconvert WITHOUT --execute (it's been executed already by you) and
# WITHOUT --no-input (we want code blocks shown).
cmd = [
    sys.executable, '-m', 'nbconvert',
    '--to', 'html',
    '--embed-images',
    '--output', str(out_html.absolute()),
    this_nb,
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
else:
    print(f'\n✓ Report written to: {out_html.resolve()}')
    size_kb = out_html.stat().st_size / 1024 if out_html.exists() else 0
    print(f'  size: {size_kb:.1f} kB')

---
**End of report.** Files produced this run (all under `notebooks/reports/`):

- `AM_Week1_Report.html` — full report with code blocks visible
- `hourly_completeness_per_site_year.csv`
- `descriptives_pollutant_county_year.csv`
- `diurnal_profiles_all_sites.csv`
- `top5_highest_loaded_sites.csv`
- `low_completeness_sites.csv`
- `monitoring_network_map_interactive.html`
- `figures/fig01_completeness_heatmap.png`
- `figures/fig02_diurnal_profiles_all_sites.png`
- `figures/fig03_monitoring_network_map.png`

Send `AM_Week1_Report.html` to Manasa + the PI for the weekly check-in.